# Graph-Enhanced DDI Detection — Heterogeneous GNN Training

**Model:** Heterogeneous GraphSAGE + Enhanced Link Predictor 
**Dataset:** DrugBank — 4,795 drugs, 824,249 DDI pairs, 2,708 human proteins  
**Loss:** nnPU Learning (Kiryo et al. 2017), prior π = 0.0717  
**Best results:** Test AUROC 0.9738 | Test AUPR 0.9589

### Architecture summary
- **Encoder:** 3-layer HeteroConv (SAGEConv), edge types: `ddi`, `targets`, `rev_targets`
- **Decoder:** EnhancedLinkPredictor — pools shared DDI neighbours and shared protein targets  
  for each pair before the MLP (NCN: Wang et al., ICLR 2024)
- **Graph:** Drug nodes (980-dim) + Protein nodes (5-dim) + drug→protein edges (21,574)

In [ ]:
#Imports
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
#Config
config = {
    # model architecture
    "drugInChannels": 980, #PubMedBERT (768) + structural (212)
    "proteinInChannels": 5, #4 role flags + log-degree
    "hiddenChannels": 256,
    "outChannels": 64,
    "predictorHidden": 128,

    #training
    "lr": 0.001,
    "epochs": 500,
    "dropout": 0.3,
    "maskRatio": 0.2,

    #nnPU prior = observed positive rate
    "prior": 824_249/(4795*4794/2), # ≈ 0.0717

    #early stopping
    "earlyStoppingPatience": 30,
    "earlyStoppingEnabled": True,

    #data split
    "valRatio": 0.1,
    "testRatio": 0.1,
}

print(f"Prior π : {config['prior']:.6f}")

In [ ]:
#Load heterogeneous graph
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

GRAPH_DIR = os.path.join(os.getcwd(), "data", "step4_graph")

data = torch.load(
    os.path.join(GRAPH_DIR, "hetero_ddi_graph.pt"),
    map_location=device,
    weights_only=False,
)

print(f"Drug nodes : {data['drug'].num_nodes:,}")
print(f"Protein nodes : {data['protein'].num_nodes:,}")
print(f"DDI edges (x2) : {data['drug','ddi','drug'].edge_index.shape[1]:,}")
print(f"Drug→protein : {data['drug','targets','protein'].edge_index.shape[1]:,}")
print(f"Drug feature dim : {data['drug'].x.shape[1]}")
print(f"Protein feat dim : {data['protein'].x.shape[1]}")

#Strip list metadata (NeighborLoader can only index-select tensors)
drug_ids_list = data["drug"].drugbank_ids
drug_names_list = data["drug"].drug_names
protein_ids_list = data["protein"].polypeptide_ids
del data["drug"].drugbank_ids
del data["drug"].drug_names
del data["protein"].polypeptide_ids

In [ ]:
#Masked positives + train/val/test split
from torch_geometric.data import Data as HomoData

def applyMaskedPositives(data, maskRatio=0.2):
    ddi_edges = data["drug", "ddi", "drug"].edge_index
    numEdges = ddi_edges.shape[1]
    numMasked = int(numEdges * maskRatio)
    perm = torch.randperm(numEdges)
    maskedIdx = perm[:numMasked]
    trainingIdx = perm[numMasked:]
    trainingEdges = ddi_edges[:, trainingIdx]
    maskedEdges = ddi_edges[:, maskedIdx]
    trainingData = data.clone()
    trainingData["drug", "ddi", "drug"].edge_index = trainingEdges
    print(f"Total DDI edges : {numEdges:,}")
    print(f"Training edges : {trainingIdx.shape[0]:,}")
    print(f"Masked (val pos) : {maskedIdx.shape[0]:,}")
    return trainingData, maskedEdges

trainingData, maskedEdges = applyMaskedPositives(data.cpu(), maskRatio=config["maskRatio"])

homo_for_split = HomoData(
    x=trainingData["drug"].x,
    edge_index=trainingData["drug", "ddi", "drug"].edge_index,
    num_nodes=trainingData["drug"].num_nodes,
)
transform = RandomLinkSplit(
    num_val=config["valRatio"],
    num_test=config["testRatio"],
    is_undirected=True,
    add_negative_train_samples=False,
)
trainHomo, valHomo, testHomo = transform(homo_for_split)

def buildSplit(evalHomo, trainEdgeIndex, fullHetero):
    d = fullHetero.clone()
    d["drug", "ddi", "drug"].edge_index = trainEdgeIndex
    d["drug", "ddi", "drug"].edge_label_index = evalHomo.edge_label_index
    d["drug", "ddi", "drug"].edge_label = evalHomo.edge_label
    return d

trainData = buildSplit(trainHomo, trainHomo.edge_index, trainingData).to(device)
valData = buildSplit(valHomo, trainHomo.edge_index, trainingData).to(device)
testData = buildSplit(testHomo, trainHomo.edge_index, trainingData).to(device)
maskedEdges = maskedEdges.to(device)

print(f"\nTrain DDI edges : {trainData['drug','ddi','drug'].edge_index.shape[1]:,}")
print(f"Val edges : {valData['drug','ddi','drug'].edge_label_index.shape[1]:,}")
print(f"Test edges  : {testData['drug','ddi','drug'].edge_label_index.shape[1]:,}")
print(f"Masked edges : {maskedEdges.shape[1]:,}")

In [ ]:
#NeighborLoader
from torch_geometric.loader import NeighborLoader

trainLoader = NeighborLoader(
    trainData,
    num_neighbors={
        ("drug", "ddi", "drug"): [25, 15, 10],
        ("drug", "targets", "protein"): [10,  5,  3],
        ("protein", "rev_targets", "drug"): [10,  5,  3],
    },
    batch_size=512,
    input_nodes=("drug", None),
    shuffle=True,
    num_workers=0,
)

print(f"Batches per epoch: {len(trainLoader)}")

## Model Definition

### Encoder
3-layer HeteroConv with SAGEConv on three edge types.  
Returns both drug embeddings (used for prediction) and protein embeddings (used for CN pooling).

### Decoder - EnhancedLinkPredictor
Inspired by NCN (Wang et al., ICLR 2024): rather than simply concatenating endpoint embeddings,
the decoder also pools embeddings of **shared DDI neighbours** and **shared protein targets**,
giving the MLP direct access to the mechanistic context of each pair.

MLP input = `[z_u | z_v | mean(z_shared_ddi) | mean(z_shared_protein)]`, so 256-dim total.

In [ ]:
#Encoder
class HeteroGraphSAGEEncoder(nn.Module):
    def __init__(self, drug_in, protein_in, hidden, out, dropout):
        super().__init__()
        self.dropout = dropout

        self.conv1 = HeteroConv({
            ("drug", "ddi", "drug"): SAGEConv(drug_in, hidden),
            ("drug", "targets", "protein"): SAGEConv((drug_in, protein_in), hidden),
            ("protein", "rev_targets", "drug"): SAGEConv((protein_in, drug_in), hidden),
        }, aggr="sum")

        self.conv2 = HeteroConv({
            ("drug", "ddi", "drug"): SAGEConv(hidden, hidden),
            ("drug", "targets", "protein"): SAGEConv(hidden, hidden),
            ("protein", "rev_targets", "drug"): SAGEConv(hidden, hidden),
        }, aggr="sum")

        self.conv3 = HeteroConv({
            ("drug", "ddi", "drug"): SAGEConv(hidden, out),
            ("drug", "targets", "protein"): SAGEConv(hidden, out),
            ("protein", "rev_targets", "drug"): SAGEConv(hidden, out),
        }, aggr="sum")

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: F.relu(v) for k, v in x_dict.items()}
        x_dict = {k: F.dropout(v, p=self.dropout, training=self.training)
                  for k, v in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        x_dict = {k: F.relu(v) for k, v in x_dict.items()}
        x_dict = {k: F.dropout(v, p=self.dropout, training=self.training)
                  for k, v in x_dict.items()}
        x_dict = self.conv3(x_dict, edge_index_dict)
        return x_dict["drug"], x_dict["protein"]


#Enhanced Link Predictor
class EnhancedLinkPredictor(nn.Module):
    def __init__(self, drug_dim: int = 64, protein_dim: int = 64, hidden: int = 128):
        super().__init__()
        in_dim = drug_dim*2 + drug_dim + protein_dim #256
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(hidden, 1),
        )
        self._drug_dim = drug_dim
        self._protein_dim = protein_dim

    @staticmethod
    def _sym_cn_pool(z, edge_index, src, dst, n_nodes):
        dev = z.device
        adj = torch.zeros(n_nodes, n_nodes, dtype=torch.bool, device=dev)
        adj[edge_index[0], edge_index[1]] = True
        adj[edge_index[1], edge_index[0]] = True
        shared = (adj[src] & adj[dst]).float()
        count = shared.sum(dim=1, keepdim=True).clamp(min=1.0)
        return (shared @ z)/count

    @staticmethod
    def _bipartite_cn_pool(z_protein, dp_edge_index, src, dst, n_drugs, n_proteins):
        dev = z_protein.device
        dp = torch.zeros(n_drugs, n_proteins, dtype=torch.bool, device=dev)
        dp[dp_edge_index[0], dp_edge_index[1]] = True
        shared = (dp[src] & dp[dst]).float()
        count = shared.sum(dim=1, keepdim=True).clamp(min=1.0)
        return (shared @ z_protein)/count

    def forward(self, z_drug, z_protein, ddi_ei, dp_ei, pred_ei):
        src, dst = pred_ei[0], pred_ei[1]
        n_d = z_drug.shape[0]
        n_p = z_protein.shape[0]
        cn_ddi = self._sym_cn_pool(z_drug, ddi_ei, src, dst, n_d)
        cn_prot = self._bipartite_cn_pool(z_protein, dp_ei, src, dst, n_d, n_p)
        pair = torch.cat([z_drug[src], z_drug[dst], cn_ddi, cn_prot], dim=-1)
        return torch.sigmoid(self.mlp(pair).squeeze(-1))

    def predict_pair(self, z_drug, z_protein, ddi_adj, dp_adj, idx_a, idx_b):
        self.eval()
        dev = z_drug.device
        zero_d = torch.zeros(self._drug_dim,    device=dev)
        zero_p = torch.zeros(self._protein_dim, device=dev)
        cn_mask = ddi_adj[idx_a] & ddi_adj[idx_b]
        cn_ddi = z_drug[cn_mask].mean(0) if cn_mask.any() else zero_d
        sp_mask = dp_adj[idx_a] & dp_adj[idx_b]
        cn_prot = z_protein[sp_mask].mean(0) if sp_mask.any() else zero_p
        vec = torch.cat([z_drug[idx_a], z_drug[idx_b], cn_ddi, cn_prot])
        return torch.sigmoid(self.mlp(vec.unsqueeze(0))).item()


#Full model
class DDIModel(nn.Module):
    def __init__(self, drug_in, protein_in, hidden, out, predictor_hidden, dropout):
        super().__init__()
        self.encoder   = HeteroGraphSAGEEncoder(drug_in, protein_in, hidden, out, dropout)
        self.predictor = EnhancedLinkPredictor(drug_dim=out, protein_dim=out, hidden=predictor_hidden)

    def forward(self, x_dict, edge_index_dict, predEdgeIndex):
        z_drug, z_protein = self.encoder(x_dict, edge_index_dict)
        ddi_ei = edge_index_dict[("drug", "ddi", "drug")]
        dp_ei  = edge_index_dict[("drug", "targets", "protein")]
        return self.predictor(z_drug, z_protein, ddi_ei, dp_ei, predEdgeIndex)


#Instantiate and inspect
model = DDIModel(
    drug_in = config["drugInChannels"],
    protein_in = config["proteinInChannels"],
    hidden = config["hiddenChannels"],
    out = config["outChannels"],
    predictor_hidden = config["predictorHidden"],
    dropout = config["dropout"],
).to(device)

print(model)
print(f"\nTotal parameters : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#nnPU Loss
class nnPULoss(nn.Module):
    def __init__(self, prior, posLossFn=None, negLossFn=None):
        super().__init__()
        self.prior = prior
        self.posLossFn = posLossFn or nn.BCELoss()
        self.negLossFn = negLossFn or nn.BCELoss()

    def forward(self, posScores, unlabeledScores):
        posRisk = self.prior * self.posLossFn(posScores, torch.ones_like(posScores))
        unlabeledRisk  = self.negLossFn(unlabeledScores, torch.zeros_like(unlabeledScores))
        correctionRisk = self.prior * self.negLossFn(posScores, torch.zeros_like(posScores))
        negRisk = torch.clamp(unlabeledRisk - correctionRisk, min=0)
        return posRisk + negRisk


optimizer = optim.Adam(model.parameters(), lr=config["lr"])
criterion = nnPULoss(prior=config["prior"])
print(f"Prior π : {config['prior']:.6f}")

In [ ]:
#Evaluation functions
def evaluate(model, data):
    model.eval()
    with torch.no_grad():
        scores = model(
            data.x_dict,
            data.edge_index_dict,
            data["drug", "ddi", "drug"].edge_label_index,
        ).cpu().numpy()
        labels = data["drug", "ddi", "drug"].edge_label.cpu().numpy()
    return roc_auc_score(labels, scores), average_precision_score(labels, scores)


def evaluateMaskedRecovery(model, trainingData, maskedEdges, numNegSamples=1000):
    model.eval()
    with torch.no_grad():
        posScores = model(
            trainingData.x_dict,
            trainingData.edge_index_dict,
            maskedEdges,
        ).cpu().numpy()
        ddi_train = trainingData["drug", "ddi", "drug"].edge_index
        negEdgeIndex = negative_sampling(
            edge_index=ddi_train,
            num_nodes=trainingData["drug"].num_nodes,
            num_neg_samples=numNegSamples,
        )
        negScores = model(
            trainingData.x_dict,
            trainingData.edge_index_dict,
            negEdgeIndex,
        ).cpu().numpy()
    scores = np.concatenate([posScores, negScores])
    labels = np.concatenate([np.ones(len(posScores)), np.zeros(len(negScores))])
    recoveryAUROC = roc_auc_score(labels, scores)
    print(f"Masked Positive Recovery AUROC: {recoveryAUROC:.4f}")
    return recoveryAUROC

In [ ]:
#Training step
def trainOneEpochPU(model, loader, optimizer, criterion):
    model.train()
    totalLoss = 0
    numBatches = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        posEdgeIndex = batch["drug", "ddi", "drug"].edge_index
        unlabeledEdgeIndex = negative_sampling(
            edge_index=posEdgeIndex,
            num_nodes=batch["drug"].num_nodes,
            num_neg_samples=posEdgeIndex.shape[1],
        ).to(device)
        posScores = model(batch.x_dict, batch.edge_index_dict, posEdgeIndex)
        unlabeledScores = model(batch.x_dict, batch.edge_index_dict, unlabeledEdgeIndex)
        loss = criterion(posScores, unlabeledScores)
        loss.backward()
        optimizer.step()
        totalLoss += loss.item()
        numBatches += 1
    return totalLoss/numBatches

In [ ]:
#Training loop
def trainLoopPU(model, trainData, valData, optimizer, criterion, config):
    bestValAUROC  = 0
    bestModelState = None
    epochsWithoutImprovement = 0

    epochBar = tqdm(range(1, config["epochs"] + 1), desc="Training", unit="epoch")
    for epoch in epochBar:
        loss = trainOneEpochPU(model, trainLoader, optimizer, criterion)
        epochBar.set_postfix(loss=f"{loss:.4f}")

        if epoch%10 == 0:
            valAUROC, valAUPR = evaluate(model, valData)
            recoveryAUROC = evaluateMaskedRecovery(
                model, trainData, maskedEdges, numNegSamples=1000)

            epochBar.set_postfix(
                loss=f"{loss:.4f}",
                valAUROC=f"{valAUROC:.4f}",
                valAUPR=f"{valAUPR:.4f}",
                recovery=f"{recoveryAUROC:.4f}",
            )
            print(f"\nEpoch {epoch:03d} | Loss: {loss:.4f} | "
                  f"Val AUROC: {valAUROC:.4f} | Val AUPR: {valAUPR:.4f} | "
                  f"Recovery: {recoveryAUROC:.4f}")

            if valAUROC > bestValAUROC:
                bestValAUROC = valAUROC
                bestModelState = {k: v.clone() for k, v in model.state_dict().items()}
                epochsWithoutImprovement = 0
            else:
                epochsWithoutImprovement += 1

            if config["earlyStoppingEnabled"]:
                if epochsWithoutImprovement >= config["earlyStoppingPatience"]:
                    print(f"\nEarly stopping at epoch {epoch} "
                          f"(no improvement for {epochsWithoutImprovement} checks)")
                    break

    print(f"\nBest Val AUROC: {bestValAUROC:.4f}")
    return bestModelState

In [ ]:
#Run training
model = DDIModel(
    drug_in = config["drugInChannels"],
    protein_in = config["proteinInChannels"],
    hidden = config["hiddenChannels"],
    out = config["outChannels"],
    predictor_hidden = config["predictorHidden"],
    dropout = config["dropout"],
).to(device)

optimizer = optim.Adam(model.parameters(), lr=config["lr"])
criterion = nnPULoss(prior=config["prior"])

bestModelState = trainLoopPU(model, trainData, valData, optimizer, criterion, config)

#Save
SAVE_PATH = os.path.join(GRAPH_DIR, "bestHeteroModel.pt")
if bestModelState is not None:
    torch.save(bestModelState, SAVE_PATH)
    print(f"Model saved → {SAVE_PATH}")

In [ ]:
#Final test evaluation
model.load_state_dict(bestModelState)
testAUROC, testAUPR = evaluate(model, testData)

print(f"FINAL TEST RESULTS")
print(f"Test AUROC : {testAUROC:.4f}")
print(f"Test AUPR : {testAUPR:.4f}")
print()
print(f"Comparison table:")
print(f"  Homo GraphSAGE (baseline)  : AUROC 0.9615 | AUPR 0.9450")
print(f"  Hetero + EnhancedDecoder (This Model)   : AUROC {testAUROC:.4f} | AUPR {testAUPR:.4f}")